# Build the v2 censored modelling cache

This notebook builds the cumulative observed downward-movement target while retaining strict outcome horizons, right-censoring, balanced sampling, missing-log handling, day-rounding parity, and provenance checks. It must be rerun before every other v2 notebook.


In [ ]:
from __future__ import annotations
import hashlib, inspect, json, os, platform, sys
from pathlib import Path
import joblib, numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name == 'v2' or not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks' / 'v2'))
from oracle_v2_targets import cumulative_downward_movement, label_position
DATA_ROOT = PROJECT_ROOT / 'data' / 'Enrollment-Data-master'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'v2'
CACHE_ROOT = ARTIFACT_ROOT / 'cache'
MODEL_ROOT = PROJECT_ROOT / 'model'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 20260812
SESSION_ORDER = ['20229','20235','20239','20245','20249','20255','20259','20265']
SEASONS = {'fall_winter':['20229','20239','20249','20259'], 'summer':['20235','20245','20255','20265']}
FINAL_TEST = {'fall_winter':'20259', 'summer':'20265'}
DEVELOPMENT = {k:[s for s in v if s != FINAL_TEST[k]] for k,v in SEASONS.items()}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def fingerprint(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def versions():
    return {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
            'scikit_learn':sklearn.__version__,'joblib':joblib.__version__}

from joblib import Parallel, delayed
from zoneinfo import ZoneInfo
TORONTO = ZoneInfo('America/Toronto')
ROWS_PER_SESSION = 300_000
CACHE_VERSION = 4
# The Summer collector commonly stops one scheduled collection before closure.
# Forty-eight hours retains that final cadence while still censoring longer gaps.
MAX_NEGATIVE_CENSOR_GAP_HOURS = 48
TARGET_NAME = 'cumulative observed downward queue movement through the final pre-deadline observation with right-censored unresolved negatives'


## Archive provenance

In [2]:
archive_files = sorted(p for s in SESSION_ORDER for p in (DATA_ROOT / s).glob('*.json'))
if not archive_files: raise FileNotFoundError(f'No archive JSON found under {DATA_ROOT}')
provenance = {
    'created_utc': pd.Timestamp.utcnow().isoformat(), 'versions': versions(),
    'sessions': SESSION_ORDER,
    'session_hashes': {s: fingerprint([(p.name, p.stat().st_size, sha256(p)) for p in sorted((DATA_ROOT/s).glob('*.json'))]) for s in SESSION_ORDER},
}
(ARTIFACT_ROOT/'data-provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
provenance

{'created_utc': '2026-08-12T18:50:53.837161+00:00',
 'versions': {'python': '3.10.12',
  'numpy': '2.2.6',
  'pandas': '2.3.3',
  'scikit_learn': '1.7.2',
  'joblib': '1.5.3'},
 'sessions': ['20229',
  '20235',
  '20239',
  '20245',
  '20249',
  '20255',
  '20259',
  '20265'],
 'session_hashes': {'20229': 'ba74a990b148d7c2',
  '20235': '3d208c8b5af3cbd7',
  '20239': 'eda87b144db60e13',
  '20245': '0641872ab04118ce',
  '20249': 'c2e5d1ddf03850b5',
  '20255': '5e10cc5efeb9fc9a',
  '20259': '086acf4cbb732782',
  '20265': '353cfd18ce2e6787'}}

## Queue reconstruction and censoring

In [ ]:
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def campus(code): return {'3':'SCAR','5':'ERIN'}.get(code[-2], 'ARTSC')
def term(code): return {'F':'fall','S':'winter','Y':'full_year'}.get(code[-1], 'unknown')
def deadline_key(code): return 'winterWaitlistClosed' if code.endswith('S') else 'fallWaitlistClosed'
def deadline(session, code):
    bundles=read_json(DATA_ROOT/session/'AAtcconstants.json'); faculty=campus(code)
    bundle=next((b for b in bundles if b.get('faculty')==faculty), bundles[0])
    return int(bundle['importantTimestamps'][deadline_key(code)])
def capacity_at(meeting, timestamp):
    complex_cap=meeting.get('enrollmentCapComplex') or {}; cap=int(complex_cap.get('initialCap',meeting['enrollmentCap']))
    for change in sorted(complex_cap.get('capChanges',[]), key=lambda x:x['time']):
        if int(change['time']) <= timestamp: cap=int(change['newCapacity'])
    return cap

def reconstruct_section(path, meeting):
    course=read_json(path); session=path.parent.name; close=deadline(session,course['code'])
    # Zip to the shorter array. Missing lecture logs are unknown, never zero.
    count=min(len(course.get('timeIntervals',[])),len(meeting.get('enrollmentLogs',[])))
    if not count: return pd.DataFrame()
    rows=[]
    for timestamp,demand in zip(course['timeIntervals'][:count],meeting['enrollmentLogs'][:count]):
        timestamp=int(timestamp); cap=capacity_at(meeting,timestamp)
        rows.append({'session':session,'course_code':course['code'],'meeting':meeting.get('meetingNumber','UNKNOWN'),
          'timestamp':timestamp,'observed_at':pd.to_datetime(timestamp,unit='s',utc=True).tz_convert(TORONTO),
          'deadline':close,'capacity':cap,'demand':int(demand),'waitlist':max(int(demand)-cap,0),
          'campus':campus(course['code']),'term':term(course['code'])})
    raw=pd.DataFrame(rows).sort_values('timestamp').reset_index(drop=True)
    raw['_raw_index']=np.arange(len(raw))
    # Never use post-deadline values: an administratively removed queue is not clearance.
    pre=raw.loc[raw.timestamp.le(close)].copy()
    if pre.empty: return pd.DataFrame()
    terminal=pre.iloc[-1]
    terminal_gap_hours=(close-terminal.timestamp)/3600
    pre['calendar_day']=pre.observed_at.dt.date
    daily=pre.groupby('calendar_day',as_index=False).tail(1).reset_index(drop=True)
    timestamps=raw.timestamp.to_numpy()
    def nearest(days_back):
        targets=daily.timestamp.to_numpy()-days_back*86400
        right=np.searchsorted(timestamps,targets).clip(0,len(timestamps)-1); left=(right-1).clip(0,len(timestamps)-1)
        chosen=np.where(abs(timestamps[left]-targets)<=abs(timestamps[right]-targets),left,right)
        return raw.iloc[chosen].reset_index(drop=True)
    p3,p7=nearest(3),nearest(7)
    daily['movement_3d']=p3.waitlist.to_numpy()-daily.waitlist.to_numpy()
    daily['movement_7d']=p7.waitlist.to_numpy()-daily.waitlist.to_numpy()
    daily['capacity_changed_7d']=(p7.capacity.to_numpy()!=daily.capacity.to_numpy()).astype('int8')
    # Serving rounds to an integer day. Training uses the identical contract.
    # Match JavaScript Math.round for a non-negative interval: floor(x + 0.5).
    daily['days_to_deadline']=np.maximum(np.floor((close-daily.timestamp)/86400+.5),0).astype('int16')
    daily['terminal_waitlist']=int(terminal.waitlist)
    # Each downward step is visible queue turnover. Later arrivals do not erase it.
    drops=(raw.waitlist.shift(1)-raw.waitlist).clip(lower=0).fillna(0)
    raw['observed_downward_movement']=drops[::-1].cumsum()[::-1]-drops
    assert int(raw.observed_downward_movement.iloc[0]) == cumulative_downward_movement(raw.waitlist)
    daily['observed_downward_movement']=raw['observed_downward_movement'].iloc[daily['_raw_index'].to_numpy()].to_numpy().astype('int32')
    daily['terminal_gap_hours']=terminal_gap_hours
    daily['offering_id']=session+':'+course['code']+':'+daily.meeting.astype(str)
    return daily

def reconstruct_session(session):
    frames=[]; failures=[]
    paths=sorted(p for p in (DATA_ROOT/session).glob('*.json') if not p.name.startswith(('AA','aa')) and p.name!='constants.json')
    for path in paths:
        try:
            course=read_json(path)
            for meeting in course.get('meetings',[]):
                if meeting.get('isCancelled') or not str(meeting.get('meetingNumber','')).startswith('LEC'): continue
                frame=reconstruct_section(path,meeting)
                if not frame.empty: frames.append(frame)
        except Exception as exc: failures.append((path.name,repr(exc)))
    if failures: raise RuntimeError(f'{session}: reconstruction failures {failures[:10]}')
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()


## Offering/day-stratified position sampling

In [ ]:
def sample_positions(daily, budget=ROWS_PER_SESSION):
    eligible=daily.loc[daily.waitlist.gt(0) & (daily.terminal_gap_hours.le(MAX_NEGATIVE_CENSOR_GAP_HOURS) | daily.observed_downward_movement.gt(0))].copy()
    strata=list(eligible.groupby(['offering_id','calendar_day'],sort=True))
    # The row budget is a target, never a reason to discard an offering-day.
    # If strata exceed it, expand just enough to retain one rank per stratum.
    effective_budget=max(budget,len(strata))
    base,extra=divmod(effective_budget,len(strata)); rows=[]
    for i,((offering_id,day),group) in enumerate(strata):
        snapshot=group.iloc[-1]; upper=int(snapshot.waitlist); count=min(upper,base+(i<extra))
        ranks=np.unique(np.rint(np.linspace(1,upper,count)).astype(int))
        for rank in ranks:
            cleared=label_position(snapshot.observed_downward_movement, rank, snapshot.terminal_gap_hours, MAX_NEGATIVE_CENSOR_GAP_HOURS)
            # Positives are observed events. A negative is usable only when follow-up
            # reaches within the allowed terminal-gap window; otherwise its outcome is censored.
            if cleared is None: continue
            row=snapshot.to_dict(); row.update(position=rank,cleared=cleared)
            rows.append(row)
    sampled=pd.DataFrame(rows)
    # Every retained day has equal mass within its offering; every offering totals one.
    per_day=sampled.groupby(['offering_id','calendar_day'],observed=True).position.transform('count')
    days=sampled.groupby('offering_id',observed=True).calendar_day.transform('nunique')
    sampled['model_weight']=1/per_day/days
    sampled['position_to_capacity']=sampled.position/sampled.capacity.replace(0,np.nan)
    sampled['waitlist_to_capacity']=sampled.waitlist/sampled.capacity.replace(0,np.nan)
    sampled['position_to_waitlist']=sampled.position/sampled.waitlist
    sampled['days_squared']=sampled.days_to_deadline**2
    sampled['log_waitlist']=np.log1p(sampled.waitlist)
    sampled['movement_velocity_7d']=sampled.movement_7d/7
    parts=sampled.course_code.str.extract(r'^([A-Z]+)(\d)(\d{2})')
    sampled['department']=parts[0].fillna('UNKNOWN'); sampled['course_level']=parts[1].fillna('UNKNOWN')
    sampled['department_level']=sampled.department+sampled.course_level
    sampled['course_number']=(parts[0]+parts[1]+parts[2]).fillna('UNKNOWN')
    return sampled

DAILY_REQUIRED={'session','offering_id','calendar_day','waitlist','observed_downward_movement','terminal_gap_hours'}
POSITION_REQUIRED={'session','offering_id','calendar_day','position','waitlist','cleared','model_weight'}

def cache_result(session,daily,sampled):
    return session,len(daily),len(sampled),daily.offering_id.nunique(),float((daily.observed_downward_movement>0).mean())

previous_manifest_path=ARTIFACT_ROOT/'cache-manifest.json'
try: previous_manifest=json.loads(previous_manifest_path.read_text())
except (FileNotFoundError,json.JSONDecodeError): previous_manifest={}

def valid_cache_pair(session,daily_path,rows_path):
    source_is_current=(previous_manifest.get('cache_version')==CACHE_VERSION and previous_manifest.get('target')==TARGET_NAME and
      previous_manifest.get('source_session_hashes',{}).get(session)==provenance['session_hashes'][session])
    if not source_is_current or not daily_path.exists() or not rows_path.exists(): return None
    try:
        daily=pd.read_pickle(daily_path); sampled=pd.read_pickle(rows_path)
        if daily.empty or sampled.empty: return None
        if not DAILY_REQUIRED.issubset(daily.columns) or not POSITION_REQUIRED.issubset(sampled.columns): return None
        if set(daily.session.astype(str).unique())!={session} or set(sampled.session.astype(str).unique())!={session}: return None
        if not sampled.position.le(sampled.waitlist).all() or not sampled.cleared.isin([0,1]).all(): return None
        if not np.allclose(sampled.groupby('offering_id').model_weight.sum(),1): return None
        return daily,sampled
    except Exception: return None

def build(session):
    daily_path=CACHE_ROOT/f'{session}-daily-v{CACHE_VERSION}.pkl'; rows_path=CACHE_ROOT/f'{session}-positions-v{CACHE_VERSION}.pkl'
    cached=valid_cache_pair(session,daily_path,rows_path)
    if cached is not None:
        print(f'{session}: valid cache found, skipping reconstruction',flush=True)
        return cache_result(session,*cached)
    print(f'{session}: reconstructing archive',flush=True)
    daily=reconstruct_session(session); sampled=sample_positions(daily)
    daily_tmp=daily_path.with_suffix('.tmp.pkl'); rows_tmp=rows_path.with_suffix('.tmp.pkl')
    daily.to_pickle(daily_tmp); sampled.to_pickle(rows_tmp)
    daily_tmp.replace(daily_path); rows_tmp.replace(rows_path)
    return cache_result(session,daily,sampled)

workers=max(1,min(int(os.environ.get('ORACLE_CACHE_WORKERS','23')),len(SESSION_ORDER)))
results=Parallel(n_jobs=workers,backend='loky',verbose=10)(joblib.delayed(build)(s) for s in SESSION_ORDER)
pd.DataFrame(results,columns=['session','daily_rows','position_rows','offerings','days_with_observed_movement_share'])


## Integrity, provenance, and handoff

In [ ]:
manifest={'cache_version':CACHE_VERSION,'target':TARGET_NAME,
 'negative_censor_gap_hours_max':MAX_NEGATIVE_CENSOR_GAP_HOURS,'sampling':'offering/day-stratified deterministic rank grid',
 'final_test':FINAL_TEST,'versions':versions(),'source_session_hashes':provenance['session_hashes'],
 'files':{p.name:sha256(p) for p in sorted(CACHE_ROOT.glob(f'*-v{CACHE_VERSION}.pkl'))}}
for session in SESSION_ORDER:
    frame=pd.read_pickle(CACHE_ROOT/f'{session}-positions-v{CACHE_VERSION}.pkl')
    assert frame.position.le(frame.waitlist).all() and frame.cleared.isin([0,1]).all()
    assert np.allclose(frame.groupby('offering_id').model_weight.sum(),1)
    daily=pd.read_pickle(CACHE_ROOT/f'{session}-daily-v{CACHE_VERSION}.pkl')
    assert daily.terminal_gap_hours.ge(0).all()
(ARTIFACT_ROOT/'cache-manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
label_audit=[]
for session in SESSION_ORDER:
    frame=pd.read_pickle(CACHE_ROOT/f'{session}-positions-v{CACHE_VERSION}.pkl'); daily=pd.read_pickle(CACHE_ROOT/f'{session}-daily-v{CACHE_VERSION}.pkl')
    label_audit.append({'session':session,'weighted_positive_rate':float(np.average(frame.cleared,weights=frame.model_weight)),'negative_offerings':int(frame.loc[frame.cleared.eq(0),'offering_id'].nunique()),'offerings_within_24h':int(daily.groupby('offering_id').terminal_gap_hours.first().le(24).sum()),'offerings_within_48h':int(daily.groupby('offering_id').terminal_gap_hours.first().le(48).sum())})
manifest['label_audit']=label_audit
(ARTIFACT_ROOT/'cache-manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
pd.DataFrame(label_audit)


## Findings to record after running

- Report the final pre-deadline observation-gap distribution and the number of unresolved negative rows removed as censored.
- Report the share of offering-days with at least one observed downward queue step.
- Confirm every labelable offering/day survives sampling and every retained offering has total model weight 1.
- Keep the latest completed Fall/Winter and Summer evaluation sessions out of Notebooks 2 and 3.
